# EDA por Aluno — AEEB 2025 (Fase 3)

Base: `data/processed/dataset_aluno.parquet` (~1,94M alunos presentes + contexto municipal).
Inclui a evidência visual do leakage F1 (proficiência ≥ 743 ⇔ alfabetizado) e o ICC municipal.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import DATA_PROCESSED, DATA_RAW, IMAGES_DIR, SEED
from src.visualization.plots import save_fig

df = pd.read_parquet(DATA_PROCESSED / 'dataset_aluno.parquet')
df.shape

(1943034, 62)

## 1. Taxa de alfabetização por rede, UF e região

In [2]:
dep_map = {1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada'}
df['rede'] = df['tp_dependencia'].astype(int).map(dep_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
df.groupby('rede', observed=True)['in_alfabetizado'].mean().sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Taxa de alfabetização por rede (alunos presentes)')
axes[0].set_xlim(0, 1)
df.groupby('regiao')['in_alfabetizado'].mean().reindex(['N','NE','CO','SE','S']).plot.bar(ax=axes[1], color='darkorange')
axes[1].set_title('Por região'); axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
save_fig(fig, IMAGES_DIR / 'eda_aluno_01_taxa_rede_regiao.png')
df.groupby('sg_uf')['in_alfabetizado'].mean().sort_values().round(3)

sg_uf
SE    0.485
RN    0.492
RS    0.536
BA    0.558
RR     0.57
AM    0.573
PA    0.584
AP    0.603
RJ    0.606
SP    0.609
TO     0.61
SC    0.639
AL    0.642
MS    0.658
PE    0.661
AC    0.676
MA    0.693
PB    0.706
MG    0.743
MT    0.752
RO     0.76
PI     0.77
ES    0.776
GO     0.79
PR      0.8
CE    0.836
Name: in_alfabetizado, dtype: Float64

## 2. Evidência F1 — proficiência determina o target (leakage)

Histograma de `VL_PROFICIENCIA_LP` colorido por `IN_ALFABETIZADO`, com linha de corte em 743.
**Figura obrigatória** do argumento anti-leakage. Usa amostra da Bronze (query única, só para o gráfico).

In [3]:
from google.cloud import bigquery
from src.config import GCP_PROJECT_ID, TBL_ALUNO

sql = f"""
SELECT VL_PROFICIENCIA_LP, IN_ALFABETIZADO
FROM `{TBL_ALUNO}`
WHERE IN_PRESENCA_LP = 1 AND IN_ALFABETIZADO IS NOT NULL
  AND MOD(ABS(FARM_FINGERPRINT(CAST(ID_ALUNO AS STRING))), 20) = 0
"""
amostra = bigquery.Client(project=GCP_PROJECT_ID).query(sql).to_dataframe()
len(amostra)

C:\Users\icaro\OneDrive\Área de Trabalho\Projetos\FIAP\Desafio3\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


98788

In [4]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for val, cor, lab in [(0, 'indianred', 'Não alfabetizado'), (1, 'seagreen', 'Alfabetizado')]:
    sub = amostra[amostra['IN_ALFABETIZADO'] == val]['VL_PROFICIENCIA_LP']
    ax.hist(sub, bins=60, alpha=0.6, color=cor, label=f'{lab} (n={len(sub):,})')
ax.axvline(743, color='black', ls='--', lw=2, label='corte = 743')
ax.set_title('VL_PROFICIENCIA_LP × IN_ALFABETIZADO — separação perfeita (leakage)')
ax.set_xlabel('Proficiência LP'); ax.legend()
save_fig(fig, IMAGES_DIR / 'eda_aluno_02_leakage_proficiencia.png')
amostra.groupby('IN_ALFABETIZADO')['VL_PROFICIENCIA_LP'].agg(['min','max','count'])

,min,max,count
IN_ALFABETIZADO,,,
0,579.821203,742.999920,33112
1,743.000220,904.022787,65497


## 3. ICC — quanto da variância está entre municípios?

ICC de um modelo nulo (só intercepto por município) sobre o target binário.
Quanto maior, mais o contexto municipal explica — e maior o teto do Modelo A.

In [5]:
g = df.groupby('co_municipio')['in_alfabetizado']
n_bar = g.count().mean()
grand = df['in_alfabetizado'].mean()
msb = (g.count() * (g.mean() - grand) ** 2).sum() / (g.ngroups - 1)
msw = ((g.count() - 1) * g.var()).sum() / (len(df) - g.ngroups)
icc = (msb - msw) / (msb + (n_bar - 1) * msw)
print(f'ICC municipal ≈ {icc:.3f}')
print(f'Interpretação: ~{100*icc:.0f}% da variância do target está ENTRE municípios;')
print(f'o restante ({100*(1-icc):.0f}%) é intra-município (escola/família/individual) — não observado.')


ICC municipal ≈ 0.081
Interpretação: ~8% da variância do target está ENTRE municípios;
o restante (92%) é intra-município (escola/família/individual) — não observado.


## 4. Taxa por faixa de IDHM-E do município

In [6]:
df['idhm_e_bin'] = pd.qcut(df['idhm_e'], 5, labels=['Q1 (baixo)','Q2','Q3','Q4','Q5 (alto)'])
fig, ax = plt.subplots(figsize=(7, 4))
df.groupby('idhm_e_bin', observed=True)['in_alfabetizado'].mean().plot.bar(ax=ax, color='teal')
ax.set_title('Taxa de alfabetização por quintil de IDHM-Educacional municipal')
ax.set_ylim(0, 1); ax.tick_params(axis='x', rotation=0)
save_fig(fig, IMAGES_DIR / 'eda_aluno_03_taxa_idhme.png')